# Simulador de Pregao — Gerador de Codigo de Trading com IA

Exercicio da **Semana 4** do curso LLM Engineering. O tema da semana e **Geracao de Codigo** — usar LLMs para gerar codigo funcional.

Neste notebook, vamos usar a OpenAI para **gerar funcoes de estrategia de trading em Python** que operam sobre uma API simulada de bolsa de valores brasileira. O diferencial: alem de gerar codigo, o notebook **executa** as estrategias geradas e mostra o P&L (lucro/prejuizo) simulado.

## O que voce vai aprender

- Como construir prompts eficazes para geracao de codigo com LLMs
- Como criar um ambiente de execucao restrito e seguro para rodar codigo gerado por IA
- Como calcular P&L simulado a partir de ordens de compra/venda
- Como integrar tudo em uma interface Gradio interativa com tema Brasil

In [ ]:
import os
import random
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key detectada: {openai_api_key[:8]}...")
else:
    print("OpenAI API Key nao encontrada — configure no .env")

openai = OpenAI()

MODELOS = {"gpt-4o-mini": "gpt-4o-mini", "gpt-4o": "gpt-4o"}

## API de Trading Simulada

O ambiente de trading simulado disponibiliza para as estrategias geradas:

- **`tickers`**: lista de 10 acoes reais da B3 (PETR4, VALE3, ITUB4, etc.)
- **`prices`**: dicionario `{ticker: [preco_dia0, preco_dia1, ...]}` com 30 dias de precos simulados (mais recente primeiro)
- **`Order(ticker, quantity)`**: classe que representa uma ordem de compra (quantidade positiva) ou venda (quantidade negativa)

Os precos sao gerados via random walk com valores realistas para cada acao.

In [ ]:
# Acoes da B3
tickers = ["PETR4", "VALE3", "ITUB4", "BBDC4", "WEGE3", "ABEV3", "RENT3", "MGLU3", "SUZB3", "BBAS3"]

# Precos base realistas (R$) para cada acao
_precos_base = {
    "PETR4": 36.0, "VALE3": 68.0, "ITUB4": 32.0, "BBDC4": 14.0, "WEGE3": 42.0,
    "ABEV3": 13.0, "RENT3": 48.0, "MGLU3": 8.0, "SUZB3": 55.0, "BBAS3": 28.0,
}

def _gerar_precos(ticker, dias=30):
    """Gera serie de precos via random walk a partir do preco base."""
    random.seed(hash(ticker))
    preco = _precos_base[ticker]
    serie = [round(preco, 2)]
    for _ in range(dias - 1):
        variacao = random.gauss(0, preco * 0.02)  # ~2% volatilidade diaria
        preco = max(0.50, preco + variacao)
        serie.append(round(preco, 2))
    serie.reverse()  # mais recente primeiro
    return serie

# Dicionario de precos: {ticker: [preco_hoje, preco_ontem, ...]}
precos = {ticker: _gerar_precos(ticker) for ticker in tickers}

# Classe Order
class Ordem:
    def __init__(self, ticker, quantidade):
        self.ticker = ticker
        self.quantidade = quantidade

    def __repr__(self):
        direcao = "COMPRA" if self.quantidade > 0 else "VENDA"
        return f"Ordem({direcao} {abs(self.quantidade)}x {self.ticker})"

# Amostra dos dados
print("Precos simulados (primeiros 5 dias):\n")
for t in tickers[:3]:
    print(f"  {t}: {precos[t][:5]}")
print(f"  ...")
print(f"\nTotal: {len(tickers)} acoes, {len(precos[tickers[0]])} dias cada")

In [ ]:
# Funcao de estrategia exemplo (string que sera enviada no prompt)
# Nomes em ingles no prompt para melhor qualidade de geracao

EXEMPLO_ESTRATEGIA = """
# tickers is a list of stock tickers (strings) from the Brazilian stock exchange B3
import tickers

# prices is a dict; the key is a ticker and the value is a list of historic prices (floats), today first
import prices

# Order represents a decision to buy or sell a quantity of a ticker
# Order("PETR4", 100) -> buy 100 shares of PETR4
# Order("VALE3", -50) -> sell/short 50 shares of VALE3
import Order

import random
import numpy as np

def strategy1():
    # Buy the top performing stock over the last 5 days based on price change
    performance = {}
    for ticker in tickers:
        if len(prices[ticker]) >= 5:
            change = (prices[ticker][0] - prices[ticker][4]) / prices[ticker][4]
            performance[ticker] = change
    best_ticker = max(performance, key=performance.get)
    return [Order(best_ticker, 100)]
"""

print("Exemplo de estrategia carregado:")
print(EXEMPLO_ESTRATEGIA)

In [ ]:
# Demo: executar a estrategia exemplo diretamente no notebook
# Isso ajuda a entender a API antes de gerar novas estrategias

def strategy1():
    """Compra a acao com melhor desempenho nos ultimos 5 dias."""
    performance = {}
    for ticker in tickers:
        if len(precos[ticker]) >= 5:
            change = (precos[ticker][0] - precos[ticker][4]) / precos[ticker][4]
            performance[ticker] = change
    best_ticker = max(performance, key=performance.get)
    return [Ordem(best_ticker, 100)]

# Executar e mostrar resultado
ordens = strategy1()
print("Ordens geradas pela strategy1():")
for o in ordens:
    print(f"  {o}")

# Calcular P&L simples: compra no preco de hoje, avalia no preco de ontem
print("\nP&L simulado:")
pnl_total = 0
for o in ordens:
    preco_hoje = precos[o.ticker][0]
    preco_ontem = precos[o.ticker][1]
    pnl = (preco_hoje - preco_ontem) * o.quantidade
    pnl_total += pnl
    print(f"  {o.ticker}: compra a R${preco_hoje:.2f}, ref. anterior R${preco_ontem:.2f} -> P&L: R${pnl:+.2f}")
print(f"\nP&L Total: R${pnl_total:+.2f}")

In [ ]:
# System prompt e user prompt (em ingles para melhor qualidade de geracao)

SYSTEM_PROMPT = """You are an expert Python developer specializing in quantitative trading strategies.
Your task is to generate Python functions that simulate trading decisions using the following API:

API DETAILS:
1. tickers: A list of stock tickers (strings) from the Brazilian stock exchange B3.
2. prices: A dictionary where the key is a stock ticker (string) and the value is a list of historical prices (floats). The list is ordered with the most recent price first. There are 30 days of data.
3. Order: A class to represent trading actions.
   - Order(ticker, quantity) creates an order:
     - Positive quantity (e.g., 100) means buying shares.
     - Negative quantity (e.g., -50) means selling/shorting shares.

INSTRUCTIONS:
- You will be provided with an example Python function demonstrating the API.
- Generate 5 additional Python functions, each implementing a unique trading strategy.
- Name them sequentially: strategy2(), strategy3(), strategy4(), strategy5(), strategy6().
- Each function must return a list of Order objects.
- Include clear comments explaining the logic.
- Return ONLY Python code. No markdown, no explanations outside of code comments.

STRATEGY IDEAS (use these or create your own):
- Momentum: trade based on recent price trends
- Mean reversion: buy oversold stocks, sell overbought ones
- Diversification: spread orders across multiple tickers
- Risk management: limit position sizes based on volatility
- Volatility analysis: trade based on price volatility patterns
"""


def user_prompt(example_function):
    return f"""Here is an example trading strategy function using the API:

{example_function}

TASK:
Based on this example and the API described in the system prompt, write 5 additional trading strategy functions named strategy2() through strategy6(). Each should implement a unique strategy and return a list of Order objects. Return only Python code."""


print("Prompts configurados.")

In [ ]:
# Funcao de geracao com streaming

def gerar_estrategias(exemplo, modelo):
    """Chama a OpenAI para gerar estrategias de trading via streaming."""
    stream = openai.chat.completions.create(
        model=MODELOS[modelo],
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt(exemplo)},
        ],
        stream=True,
    )
    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        # Strip markdown fences se o modelo incluir
        clean = reply.replace("```python\n", "").replace("```python", "").replace("```\n", "").replace("```", "")
        yield clean

## Motor de Execucao e Calculo de P&L

O codigo gerado pelo LLM e executado em um ambiente restrito via `exec()` com globals limitados. Apenas as variaveis da API de trading (`tickers`, `prices`, `Order`) e bibliotecas seguras (`numpy`, `random`) ficam disponiveis.

Apos a execucao, todas as funcoes `strategy*` encontradas sao chamadas. Para cada ordem retornada, o P&L e calculado de forma simples: **compra no preco de hoje, avalia no preco de ontem**.

In [ ]:
# Motor de execucao e calculo de P&L

def _calcular_pnl(ordens):
    """Calcula P&L simples: compra hoje (prices[0]), avalia ontem (prices[1])."""
    linhas = []
    pnl_total = 0
    for o in ordens:
        if o.ticker not in precos or len(precos[o.ticker]) < 2:
            linhas.append(f"  {o.ticker}: ticker invalido ou dados insuficientes")
            continue
        preco_hoje = precos[o.ticker][0]
        preco_ontem = precos[o.ticker][1]
        direcao = "COMPRA" if o.quantidade > 0 else "VENDA"
        pnl = (preco_hoje - preco_ontem) * o.quantidade
        pnl_total += pnl
        linhas.append(
            f"  {direcao} {abs(o.quantidade)}x {o.ticker} @ R${preco_hoje:.2f} "
            f"(ref. R${preco_ontem:.2f}) -> P&L: R${pnl:+.2f}"
        )
    linhas.append(f"\n  P&L Total: R${pnl_total:+.2f}")
    return "\n".join(linhas)


def executar_codigo_gerado(codigo_gerado):
    """Executa o codigo gerado em ambiente restrito e calcula P&L de cada estrategia."""
    if not codigo_gerado or not codigo_gerado.strip():
        return "Nenhum codigo para executar. Gere as estrategias primeiro."

    # Ambiente restrito com a API de trading
    env = {
        "__builtins__": __builtins__,
        "tickers": tickers,
        "prices": precos,
        "Order": Ordem,
        "np": np,
        "numpy": np,
        "random": random,
    }

    try:
        exec(codigo_gerado, env)
    except Exception as e:
        return f"Erro ao executar codigo gerado:\n{e}"

    # Encontrar todas as funcoes strategy* no ambiente
    estrategias = {
        nome: func for nome, func in env.items()
        if callable(func) and (nome.startswith("strategy") or nome.startswith("estrategia"))
    }

    if not estrategias:
        return "Nenhuma funcao strategy*() ou estrategia*() encontrada no codigo gerado."

    resultados = []
    for nome in sorted(estrategias.keys()):
        func = estrategias[nome]
        try:
            ordens = func()
            if not isinstance(ordens, list):
                resultados.append(f"\n{nome}(): retorno invalido (esperado lista de Order)")
                continue
            resultados.append(f"\n{nome}():")
            resultados.append(_calcular_pnl(ordens))
        except Exception as e:
            resultados.append(f"\n{nome}(): erro na execucao -> {e}")

    return "\n".join(resultados)

In [ ]:
# Demo inline (sem Gradio): gerar e executar estrategias diretamente no notebook

print("Gerando estrategias com gpt-4o-mini...\n")

codigo_gerado = ""
for chunk in gerar_estrategias(EXEMPLO_ESTRATEGIA, "gpt-4o-mini"):
    codigo_gerado = chunk

print("=== CODIGO GERADO ===\n")
print(codigo_gerado)
print("\n=== RESULTADOS DA EXECUCAO ===\n")
resultado = executar_codigo_gerado(codigo_gerado)
print(resultado)

In [ ]:
# Interface Gradio com tema Brasil (verde e dourado)

CSS_BRASIL = """
#titulo {
    text-align: center;
    font-size: 2.2em;
    font-weight: bold;
    color: #006400;
    margin-bottom: 5px;
}
#subtitulo {
    text-align: center;
    color: #666;
    margin-bottom: 15px;
}
#btn-gerar {
    background-color: #006400 !important;
    color: white !important;
    font-size: 1.3em !important;
    padding: 10px 20px !important;
    border: none !important;
    border-radius: 8px !important;
}
#btn-gerar:hover {
    background-color: #008000 !important;
}
#btn-executar {
    background-color: #DAA520 !important;
    color: white !important;
    font-size: 1.3em !important;
    padding: 10px 20px !important;
    border: none !important;
    border-radius: 8px !important;
}
#btn-executar:hover {
    background-color: #FFD700 !important;
    color: #333 !important;
}
#resultado-pnl textarea {
    background-color: #1a1a2e !important;
    color: #00ff88 !important;
    font-family: monospace !important;
    font-size: 0.95em !important;
}
#codigo-gerado textarea {
    font-family: monospace !important;
    font-size: 0.9em !important;
}
"""

with gr.Blocks(css=CSS_BRASIL, theme=gr.themes.Soft(), title="Simulador de Pregao") as ui:
    gr.Markdown("<div id='titulo'>Simulador de Pregao</div>")
    gr.Markdown("<div id='subtitulo'>Gerador de Codigo de Trading com IA — Bolsa B3</div>")

    with gr.Row():
        with gr.Column(scale=1):
            exemplo_input = gr.Textbox(
                label="Funcao Exemplo (editavel)",
                value=EXEMPLO_ESTRATEGIA,
                lines=18,
            )
            modelo_dropdown = gr.Dropdown(
                choices=list(MODELOS.keys()),
                value="gpt-4o-mini",
                label="Modelo OpenAI",
            )
        with gr.Column(scale=1):
            codigo_output = gr.Textbox(
                label="Codigo Gerado",
                lines=22,
                interactive=False,
                elem_id="codigo-gerado",
            )

    with gr.Row():
        btn_gerar = gr.Button("Gerar Estrategias", elem_id="btn-gerar")
        btn_executar = gr.Button("Executar Estrategias", elem_id="btn-executar")

    with gr.Row():
        resultado_output = gr.Textbox(
            label="Resultados — P&L Simulado",
            lines=15,
            interactive=False,
            elem_id="resultado-pnl",
        )

    btn_gerar.click(fn=gerar_estrategias, inputs=[exemplo_input, modelo_dropdown], outputs=[codigo_output])
    btn_executar.click(fn=executar_codigo_gerado, inputs=[codigo_output], outputs=[resultado_output])

print("Interface Gradio montada. Execute a proxima celula para iniciar.")

In [ ]:
ui.launch(inbrowser=True)

## Conclusao

Neste exercicio, exploramos o poder dos LLMs para **geracao de codigo funcional** no dominio de trading:

- Construimos uma **API de trading simulada** com acoes reais da B3
- Criamos prompts eficazes que geram **estrategias de trading executaveis**
- Implementamos um **motor de execucao seguro** com `exec()` e globals restritos
- Calculamos **P&L simulado** para cada estrategia gerada
- Integramos tudo em uma **interface Gradio** com tema Brasil

### Sugestoes para explorar mais

- Edite o exemplo no Gradio para guiar o LLM a gerar estrategias diferentes
- Troque o modelo para `gpt-4o` e compare a qualidade das estrategias geradas
- Adicione mais tickers ao ambiente de trading
- Implemente backtesting com multiplos dias em vez de apenas hoje/ontem
- Experimente adicionar indicadores tecnicos (RSI, MACD, Bollinger Bands) a API